In [2]:
!pip install pyttsx3


   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   ------------- -------------------------- 1/3 [comtypes]
   -------------------------- ------------- 2/3 [pyttsx3]
   ---------------------------------------- 3/3 [pyttsx3]



In [3]:
# ========================================
# AfyaMetrix Voice Assistant
# Notebook: 08_voice_assistant.ipynb
# ========================================
#
# Architecture:
#
# [User speaks] 
#      ↓
# [Browser Web Speech API converts to text]
#      ↓  
# [Text sent to /api/voice-query endpoint]
#      ↓
# [API returns data + response text]
#      ↓
# [pyttsx3 speaks the response aloud]
#      ↓
# [Dashboard updates with returned data]
#
# Why this approach?
# - Web Speech API: built into Chrome/Edge, no install needed
# - pyttsx3: works fully offline, no API key, fast response
# - Together they work on any African network condition

import pyttsx3
import requests
import json
import time
from IPython.display import display, HTML, Javascript
import ipywidgets as widgets

print("✅ Imports successful")

# Test pyttsx3 is working
try:
    engine = pyttsx3.init()
    voices = engine.getProperty('voices')
    print(f"✅ Text-to-speech ready")
    print(f"   Available voices: {len(voices)}")
    for i, voice in enumerate(voices[:3]):
        print(f"   Voice {i}: {voice.name}")
except Exception as e:
    print(f"⚠️  pyttsx3 issue: {e}")
    print("   Run: pip install pyttsx3")

✅ Imports successful
✅ Text-to-speech ready
   Available voices: 2
   Voice 0: Microsoft David Desktop - English (United States)
   Voice 1: Microsoft Zira Desktop - English (United States)


In [4]:
# ========================================
# TEXT TO SPEECH ENGINE
# ========================================

def speak(text, rate=175, volume=0.9):
    """
    Converts text to speech and plays it aloud.
    
    Parameters:
    - text: the string to speak
    - rate: words per minute (175 is natural pace)
    - volume: 0.0 to 1.0
    """
    try:
        engine = pyttsx3.init()
        
        # Set speech rate — 175 wpm sounds natural and professional
        engine.setProperty('rate', rate)
        
        # Set volume
        engine.setProperty('volume', volume)
        
        # Use female voice if available (index 1 on most Windows systems)
        voices = engine.getProperty('voices')
        if len(voices) > 1:
            engine.setProperty('voice', voices[1].id)
        
        engine.say(text)
        engine.runAndWait()
        engine.stop()
        
        return True
        
    except Exception as e:
        print(f"Speech error: {e}")
        return False


# Test it
print("🔊 Testing voice output...")
print("   You should hear: 'AfyaMetrix voice assistant is ready'")
speak("AfyaMetrix voice assistant is ready. "
      "I can help you monitor disease outbreaks across Africa.")
print("✅ Voice output working")

🔊 Testing voice output...
   You should hear: 'AfyaMetrix voice assistant is ready'
✅ Voice output working


In [5]:
# ========================================
# QUERY PROCESSOR
# ========================================
# Sends text queries to the API and formats responses

API_BASE = "http://127.0.0.1:8000"

def process_voice_query(query_text):
    """
    Takes transcribed speech text, sends to API,
    returns formatted response for both display and speech.
    
    Parameters:
    - query_text: string from speech recognition
    
    Returns: dict with response text and data
    """
    
    print(f"\n🎤 Query received: '{query_text}'")
    
    try:
        # Send to voice query endpoint
        response = requests.get(
            f"{API_BASE}/api/voice-query",
            params={"q": query_text},
            timeout=5
        )
        
        if response.status_code == 200:
            result = response.json()
            intent   = result.get('intent', 'unknown')
            response_text = result.get('response', 'No response available')
            data     = result.get('data', [])
            
            print(f"   Intent detected: {intent}")
            print(f"   Response: {response_text}")
            
            # Speak the response
            speak(response_text)
            
            return {
                "success":  True,
                "intent":   intent,
                "response": response_text,
                "data":     data
            }
        else:
            error_msg = "Sorry, I could not process that query."
            speak(error_msg)
            return {"success": False, "response": error_msg}
            
    except requests.exceptions.ConnectionError:
        msg = "API server is not running. Please start it first."
        print(f"❌ {msg}")
        speak(msg)
        return {"success": False, "response": msg}
    except Exception as e:
        msg = "An error occurred processing your query."
        print(f"❌ Error: {e}")
        return {"success": False, "response": msg}


# Test with sample queries
print("🧪 Testing query processor with sample queries:\n")

test_queries = [
    "show high risk regions",
    "malaria status",
    "overview"
]

for query in test_queries:
    result = process_voice_query(query)
    print(f"   ✅ Intent: {result['intent']}\n")
    time.sleep(1)  # small pause between tests

🧪 Testing query processor with sample queries:


🎤 Query received: 'show high risk regions'
   Intent detected: high_risk_regions
   Response: The 5 highest risk regions are: Somali Region, Ethiopia (risk: 71); Afar, Ethiopia (risk: 65); Diourbel, Senegal (risk: 55); Northern Province, Zambia (risk: 54); Kisangani, DRC (risk: 54).
   ✅ Intent: high_risk_regions


🎤 Query received: 'malaria status'
   Intent detected: disease_query
   Response: Malaria is most active in: Zanzibar, Tanzania; Blue Nile, Sudan; Mbeya, Tanzania.
   ✅ Intent: disease_query


🎤 Query received: 'overview'
   Intent detected: summary
   Response: AfyaMetrix status update: 0 critical regions, 2 high-risk regions across 10 countries. Latest data from 2024-12-31.
   ✅ Intent: summary



In [6]:
# ========================================
# BROWSER VOICE INPUT WIDGET
# ========================================
# This creates a widget in Jupyter that uses the
# browser's built-in microphone for speech recognition.
# Works in Chrome and Edge (not Firefox).

def create_voice_assistant_widget():
    """
    Creates an interactive voice assistant widget
    that displays inside the Jupyter notebook.
    """
    
    # HTML + JavaScript for the voice interface
    html_content = """
    <div id="afya-voice-assistant" style="
        font-family: 'Inter', sans-serif;
        background: linear-gradient(135deg, #0A7B6E, #0d5c52);
        border-radius: 16px;
        padding: 24px;
        max-width: 600px;
        color: white;
        box-shadow: 0 8px 32px rgba(0,0,0,0.3);
    ">
        <!-- Header -->
        <div style="display:flex; align-items:center; margin-bottom:20px;">
            <div style="
                width:48px; height:48px;
                background:rgba(255,255,255,0.2);
                border-radius:50%;
                display:flex; align-items:center; 
                justify-content:center;
                font-size:24px; margin-right:12px;
            ">🌍</div>
            <div>
                <h2 style="margin:0; font-size:20px;">AfyaMetrix Voice Assistant</h2>
                <p style="margin:0; opacity:0.8; font-size:13px;">
                    Pan-Africa Health Intelligence
                </p>
            </div>
            <div id="status-dot" style="
                width:12px; height:12px;
                background:#2E7D52;
                border-radius:50%;
                margin-left:auto;
                box-shadow: 0 0 8px #2E7D52;
            "></div>
        </div>
        
        <!-- Response display -->
        <div id="response-box" style="
            background:rgba(255,255,255,0.1);
            border-radius:12px;
            padding:16px;
            min-height:80px;
            margin-bottom:16px;
            font-size:15px;
            line-height:1.6;
        ">
            <span style="opacity:0.6;">
                👋 Say something like:<br>
                • "Show high risk regions"<br>
                • "Malaria status in Kenya"<br>
                • "Give me an overview"<br>
                • "Cross border alerts"
            </span>
        </div>
        
        <!-- Query display -->
        <div id="query-box" style="
            background:rgba(0,0,0,0.2);
            border-radius:8px;
            padding:10px 14px;
            margin-bottom:16px;
            font-size:13px;
            opacity:0.8;
            min-height:36px;
        ">
            <span id="query-text">Waiting for voice input...</span>
        </div>
        
        <!-- Buttons -->
        <div style="display:flex; gap:12px;">
            <button id="mic-btn" onclick="startListening()" style="
                flex:1;
                background:#F5A623;
                color:white;
                border:none;
                border-radius:10px;
                padding:14px;
                font-size:16px;
                cursor:pointer;
                font-weight:600;
                transition: all 0.2s;
            ">
                🎤 Hold to Speak
            </button>
            
            <button onclick="clearResponse()" style="
                background:rgba(255,255,255,0.15);
                color:white;
                border:none;
                border-radius:10px;
                padding:14px 18px;
                font-size:16px;
                cursor:pointer;
            ">
                🗑️
            </button>
        </div>
        
        <!-- Quick command buttons -->
        <div style="margin-top:14px;">
            <p style="margin:0 0 8px 0; font-size:12px; opacity:0.7;">
                QUICK COMMANDS:
            </p>
            <div style="display:flex; gap:8px; flex-wrap:wrap;">
                <button onclick="quickQuery('show high risk regions')" 
                    style="background:rgba(232,64,42,0.3); color:white; 
                           border:1px solid rgba(232,64,42,0.5); 
                           border-radius:6px; padding:6px 12px; 
                           font-size:12px; cursor:pointer;">
                    🔴 High Risk
                </button>
                <button onclick="quickQuery('give me an overview')" 
                    style="background:rgba(255,255,255,0.1); color:white; 
                           border:1px solid rgba(255,255,255,0.2); 
                           border-radius:6px; padding:6px 12px; 
                           font-size:12px; cursor:pointer;">
                    📊 Overview
                </button>
                <button onclick="quickQuery('cross border alerts')" 
                    style="background:rgba(74,144,217,0.3); color:white; 
                           border:1px solid rgba(74,144,217,0.5); 
                           border-radius:6px; padding:6px 12px; 
                           font-size:12px; cursor:pointer;">
                    🌍 Borders
                </button>
                <button onclick="quickQuery('malaria status')" 
                    style="background:rgba(245,166,35,0.3); color:white; 
                           border:1px solid rgba(245,166,35,0.5); 
                           border-radius:6px; padding:6px 12px; 
                           font-size:12px; cursor:pointer;">
                    🦟 Malaria
                </button>
            </div>
        </div>
    </div>
    
    <script>
    // ================================
    // SPEECH RECOGNITION SETUP
    // ================================
    // Uses browser's built-in Web Speech API
    // Works in Chrome and Edge without any API key
    
    let recognition = null;
    let isListening = false;
    
    function initRecognition() {
        if (!('webkitSpeechRecognition' in window) && 
            !('SpeechRecognition' in window)) {
            document.getElementById('response-box').innerHTML = 
                '⚠️ Speech recognition not supported in this browser. ' +
                'Please use Chrome or Edge. ' +
                'You can still use the Quick Commands buttons below.';
            return false;
        }
        
        const SpeechRecognition = window.SpeechRecognition || 
                                   window.webkitSpeechRecognition;
        recognition = new SpeechRecognition();
        
        // Settings
        recognition.continuous     = false;  // stop after one phrase
        recognition.interimResults = true;   // show partial results
        recognition.lang           = 'en-US';
        recognition.maxAlternatives = 1;
        
        // When we get a result
        recognition.onresult = function(event) {
            let transcript = '';
            let isFinal    = false;
            
            for (let i = event.resultIndex; i < event.results.length; i++) {
                transcript = event.results[i][0].transcript;
                isFinal    = event.results[i].isFinal;
            }
            
            document.getElementById('query-text').textContent = 
                (isFinal ? '✅ ' : '🎤 ') + transcript;
            
            if (isFinal) {
                stopListening();
                sendQuery(transcript);
            }
        };
        
        // Error handling
        recognition.onerror = function(event) {
            stopListening();
            let msg = 'Speech error: ' + event.error;
            if (event.error === 'not-allowed') {
                msg = '🎤 Microphone access denied. ' +
                      'Please allow microphone in browser settings.';
            } else if (event.error === 'no-speech') {
                msg = '🔇 No speech detected. Please try again.';
            }
            document.getElementById('response-box').innerHTML = msg;
        };
        
        recognition.onend = function() {
            if (isListening) stopListening();
        };
        
        return true;
    }
    
    function startListening() {
        if (!recognition && !initRecognition()) return;
        
        isListening = true;
        recognition.start();
        
        // Visual feedback
        const btn = document.getElementById('mic-btn');
        const dot = document.getElementById('status-dot');
        
        btn.style.background    = '#E8402A';
        btn.textContent         = '🔴 Listening...';
        dot.style.background    = '#E8402A';
        dot.style.boxShadow     = '0 0 12px #E8402A';
        
        document.getElementById('response-box').innerHTML = 
            '<span style="opacity:0.7;">🎤 Listening... speak now</span>';
    }
    
    function stopListening() {
        isListening = false;
        if (recognition) recognition.stop();
        
        const btn = document.getElementById('mic-btn');
        const dot = document.getElementById('status-dot');
        
        btn.style.background = '#F5A623';
        btn.textContent      = '🎤 Hold to Speak';
        dot.style.background = '#2E7D52';
        dot.style.boxShadow  = '0 0 8px #2E7D52';
    }
    
    function quickQuery(text) {
        document.getElementById('query-text').textContent = '✅ ' + text;
        sendQuery(text);
    }
    
    function clearResponse() {
        document.getElementById('response-box').innerHTML = 
            '<span style="opacity:0.6;">Ready for your next query...</span>';
        document.getElementById('query-text').textContent = 
            'Waiting for voice input...';
    }
    
    function sendQuery(queryText) {
        // Show loading state
        document.getElementById('response-box').innerHTML = 
            '<span style="opacity:0.7;">⚙️ Processing query...</span>';
        
        // Call our FastAPI endpoint
        fetch('http://127.0.0.1:8000/api/voice-query?q=' + 
              encodeURIComponent(queryText))
            .then(response => response.json())
            .then(data => {
                displayResponse(data);
            })
            .catch(error => {
                document.getElementById('response-box').innerHTML = 
                    '❌ Could not connect to AfyaMetrix API. ' +
                    'Please ensure the server is running.';
            });
    }
    
    function displayResponse(data) {
        const intent   = data.intent   || 'unknown';
        const response = data.response || 'No response';
        const results  = data.data     || [];
        
        // Intent color coding
        const intentColors = {
            'high_risk_regions': '#E8402A',
            'disease_query':     '#F5A623',
            'country_query':     '#4A90D9',
            'cross_border':      '#9B59B6',
            'summary':           '#2E7D52',
            'unknown':           '#95a5a6'
        };
        const color = intentColors[intent] || '#95a5a6';
        
        // Build results table if we have data
        let tableHTML = '';
        if (results.length > 0 && results[0].region) {
            tableHTML = '<div style="margin-top:12px; font-size:12px;">';
            results.slice(0, 5).forEach(r => {
                const risk  = r.risk_score || 0;
                const rColor = risk >= 75 ? '#E8402A' : 
                               risk >= 55 ? '#F5A623' : '#2E7D52';
                tableHTML += `
                    <div style="
                        display:flex; justify-content:space-between;
                        padding:6px 0; 
                        border-bottom:1px solid rgba(255,255,255,0.1);
                    ">
                        <span>${r.region}, ${r.country}</span>
                        <span style="color:${rColor}; font-weight:600;">
                            ${risk.toFixed(0)}/100
                        </span>
                    </div>`;
            });
            tableHTML += '</div>';
        }
        
        document.getElementById('response-box').innerHTML = `
            <div style="
                display:inline-block;
                background:${color}33;
                border:1px solid ${color}66;
                border-radius:6px;
                padding:3px 8px;
                font-size:11px;
                margin-bottom:8px;
                text-transform:uppercase;
                letter-spacing:1px;
            ">${intent.replace('_', ' ')}</div>
            <div style="font-size:15px; line-height:1.6;">
                ${response}
            </div>
            ${tableHTML}
        `;
    }
    
    // Initialize on load
    initRecognition();
    </script>
    """
    
    display(HTML(html_content))
    print("\n✅ Voice Assistant widget loaded")
    print("   Click '🎤 Hold to Speak' and allow microphone access")
    print("   Works best in Chrome or Edge browser")
    print("   Or use the Quick Commands buttons to test without voice")

# Display the widget
create_voice_assistant_widget()


✅ Voice Assistant widget loaded
   Click '🎤 Hold to Speak' and allow microphone access
   Works best in Chrome or Edge browser
   Or use the Quick Commands buttons to test without voice


In [7]:
# ========================================
# TEXT INPUT FALLBACK
# ========================================
# For environments where microphone isn't available,
# this lets you type queries and get spoken responses.
# Useful for testing and for low-bandwidth scenarios.

print("⌨️  TEXT INPUT FALLBACK")
print("Type queries below to test the voice assistant engine.\n")
print("Example queries to try:")
print("  show high risk regions")
print("  malaria in Nigeria") 
print("  cross border alerts")
print("  give me an overview")
print("  cholera status")
print("  Ethiopia health update")
print()

# Interactive test loop — run this cell, type a query, press Enter
# Type 'exit' to stop

test_queries = [
    "show high risk regions",
    "cholera in Nigeria",
    "cross border alerts",
    "give me an overview",
    "Ethiopia health update"
]

print("Running automated test of all sample queries:\n")

for query in test_queries:
    print(f"{'─' * 50}")
    result = process_voice_query(query)
    time.sleep(2)  # pause between spoken responses

print(f"\n{'─' * 50}")
print("✅ All test queries complete")
print("\nVoice Assistant is production-ready.")
print("Frontend team can call: GET /api/voice-query?q=YOUR_QUERY")

⌨️  TEXT INPUT FALLBACK
Type queries below to test the voice assistant engine.

Example queries to try:
  show high risk regions
  malaria in Nigeria
  cross border alerts
  give me an overview
  cholera status
  Ethiopia health update

Running automated test of all sample queries:

──────────────────────────────────────────────────

🎤 Query received: 'show high risk regions'
   Intent detected: high_risk_regions
   Response: The 5 highest risk regions are: Somali Region, Ethiopia (risk: 71); Afar, Ethiopia (risk: 65); Diourbel, Senegal (risk: 55); Northern Province, Zambia (risk: 54); Kisangani, DRC (risk: 54).
──────────────────────────────────────────────────

🎤 Query received: 'cholera in Nigeria'
   Intent detected: disease_query
   Response: Cholera is most active in: Somali Region, Ethiopia; Afar, Ethiopia; Diourbel, Senegal.
──────────────────────────────────────────────────

🎤 Query received: 'cross border alerts'
   Intent detected: cross_border
   Response: There are 8 activ

In [8]:
# ========================================
# VOICE ASSISTANT CAPABILITIES SUMMARY
# ========================================

print("""
╔══════════════════════════════════════════════════════╗
║         AFYAMETRIX VOICE ASSISTANT — READY           ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  SUPPORTED VOICE COMMANDS:                           ║
║                                                      ║
║  🔴 "Show high risk regions"                         ║
║     → Lists top 5 most dangerous regions             ║
║                                                      ║
║  🦟 "Malaria status" / "Cholera in Kenya"            ║
║     → Disease-specific outbreak information          ║
║                                                      ║  
║  🌍 "Cross border alerts"                            ║
║     → Shows disease spread risk across borders       ║
║                                                      ║
║  📊 "Give me an overview" / "Summary"                ║
║     → Complete system status update                  ║
║                                                      ║
║  🗺️  "Kenya status" / "Nigeria update"               ║
║     → Country-specific health intelligence           ║
║                                                      ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  TECHNICAL STACK:                                    ║
║  • Input:  Browser Web Speech API (Chrome/Edge)      ║
║  • NLP:    Keyword intent matching                   ║
║  • Data:   AfyaMetrix FastAPI backend                ║
║  • Output: pyttsx3 offline text-to-speech            ║
║  • Languages: English (expandable)                   ║
║                                                      ║
║  API ENDPOINT:                                       ║
║  GET /api/voice-query?q=show high risk regions       ║
║                                                      ║
╚══════════════════════════════════════════════════════╝
""")

print("🎯 Complete AfyaMetrix AI/ML System Status:\n")

modules = [
    ("Data Simulation Engine",      "497,080 records across 10 countries"),
    ("Anomaly Detection",           "Z-Score + Isolation Forest, 12,673 alerts"),
    ("Risk Scoring",                "0-100 score, 5-factor weighted formula"),
    ("Forecasting",                 "Prophet + Moving Average, 30-day outlook"),
    ("Clustering",                  "4 behavioral clusters, intervention plans"),
    ("Resource Allocation",         "6 resource types, proportional weighted"),
    ("Narrative Engine",            "5 languages, template-based generation"),
    ("Cross-Border Alerts",         "8 active alerts across 5 countries"),
    ("SMS Fallback Parser",         "10 disease codes, 10 country codes"),
    ("Data Quality Scorer",         "4-factor quality grading per facility"),
    ("FastAPI Backend",             "12 endpoints, live at port 8000"),
    ("Voice Assistant",             "Speech-in + Speech-out, 7 intents"),
]

for module, detail in modules:
    print(f"  ✅ {module:<30} {detail}")

print(f"\n🏆 Round 1 submission ready.")
print(f"   Share API docs with frontend: http://127.0.0.1:8000/docs")


╔══════════════════════════════════════════════════════╗
║         AFYAMETRIX VOICE ASSISTANT — READY           ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  SUPPORTED VOICE COMMANDS:                           ║
║                                                      ║
║  🔴 "Show high risk regions"                         ║
║     → Lists top 5 most dangerous regions             ║
║                                                      ║
║  🦟 "Malaria status" / "Cholera in Kenya"            ║
║     → Disease-specific outbreak information          ║
║                                                      ║  
║  🌍 "Cross border alerts"                            ║
║     → Shows disease spread risk across borders       ║
║                                                      ║
║  📊 "Give me an overview" / "Summary"                ║
║     → Complete system status update                  ║
║                               